<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/scaling_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install alignn slakonet

In [ ]:
%matplotlib inline
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms

# ALIGNN-FF (non-DGL default model via calculator)
from alignn.ff.ff import AlignnAtomwiseCalculator, default_path

# SlakoNet
from slakonet.atoms import Geometry
from slakonet.optim import get_atoms, default_model
from slakonet.main import generate_shell_dict_upto_Z65

# ── 1. Setup (GPU only) ───────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No CUDA device found — this benchmark is GPU-only."
device = "cuda"

# shared base structure: Si (JVASP-1002)
dat   = get_jid_data(jid="JVASP-1002", dataset="dft_3d")
atoms = Atoms.from_dict(dat["atoms"])
print("base cell:", atoms.composition.reduced_formula, "| atoms:", atoms.num_atoms)

MAX_ATOMS = 1100
N_REPEATS = 3

# ALIGNN model
alignn_calc = AlignnAtomwiseCalculator(path=default_path(), device=device)

# SlakoNet model
sk_model    = default_model()
shell_dict  = generate_shell_dict_upto_Z65()
kpoints     = torch.tensor([1, 1, 1])

# ── 2. Warmup ─────────────────────────────────────────────────────────────────
warm = atoms.make_supercell_matrix([2, 2, 2]).ase_converter()
warm.calc = alignn_calc
_ = warm.get_potential_energy()
torch.cuda.synchronize()

# ── 3. Scaling benchmark ──────────────────────────────────────────────────────
num_atoms, t_alignn, t_slako = [], [], []

for i in tqdm(range(1, 12), desc="Scaling (GPU)"):
    sc = atoms.make_supercell_matrix([i, i, i])
    n  = sc.num_atoms
    if n > MAX_ATOMS:
        break
    num_atoms.append(n)

    # --- ALIGNN-FF ---
    ase_sc = sc.ase_converter()
    ase_sc.calc = alignn_calc
    try:
        ts = []
        for _ in range(N_REPEATS):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            e = ase_sc.get_potential_energy()
            torch.cuda.synchronize(); ts.append(time.perf_counter() - t0)
        t_alignn.append(np.median(ts))
    except torch.cuda.OutOfMemoryError:
        print(f"  [ALIGNN OOM] N={n}"); t_alignn.append(float("nan")); torch.cuda.empty_cache()

    # --- SlakoNet ---
    geometry = Geometry.from_ase_atoms([sc.ase_converter()])
    try:
        ts = []
        for _ in range(N_REPEATS):
            with torch.no_grad():
                torch.cuda.synchronize(); t0 = time.perf_counter()
                props, success = sk_model.compute_multi_element_properties(
                    geometry=geometry, shell_dict=shell_dict,
                    kpoints=kpoints, get_energy=True, device="cuda",
                )
                torch.cuda.synchronize(); ts.append(time.perf_counter() - t0)
        t_slako.append(np.median(ts))
    except torch.cuda.OutOfMemoryError:
        print(f"  [SlakoNet OOM] N={n}"); t_slako.append(float("nan")); torch.cuda.empty_cache()

    print(f"  sc={i:2d}  N={n:5d}  ALIGNN={t_alignn[-1]:.3f}s  SlakoNet={t_slako[-1]:.3f}s")

num_atoms = np.array(num_atoms)
t_alignn  = np.array(t_alignn)
t_slako   = np.array(t_slako)

# ── 4. Summary + power-law fits ───────────────────────────────────────────────
def fit(N, t):
    m = np.isfinite(t)
    if m.sum() < 3: return None
    a, c = np.polyfit(np.log(N[m]), np.log(t[m]), 1)
    return a, np.exp(c)

print("\nnum_atoms :", num_atoms.tolist())
print("ALIGNN    :", [f"{t:.3f}" for t in t_alignn])
print("SlakoNet  :", [f"{t:.3f}" for t in t_slako])

fa = fit(num_atoms, t_alignn)
fs = fit(num_atoms, t_slako)
if fa: print(f"\nALIGNN-FF:  t ~ {fa[1]:.2e} * N^{fa[0]:.2f}")
if fs: print(f"SlakoNet:   t ~ {fs[1]:.2e} * N^{fs[0]:.2f}")

# ── 5. Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(num_atoms, t_alignn, "s--", color="tomato",   lw=2, ms=6, label="ALIGNN-FF")
ax.plot(num_atoms, t_slako,  "o-",  color="steelblue", lw=2, ms=6, label="SlakoNet")

nf = np.linspace(num_atoms.min(), num_atoms.max(), 100)
if fa: ax.plot(nf, fa[1]*nf**fa[0], ":", color="darkred",  alpha=0.6, label=f"ALIGNN $N^{{{fa[0]:.2f}}}$")
if fs: ax.plot(nf, fs[1]*nf**fs[0], ":", color="navy",     alpha=0.6, label=f"SlakoNet $N^{{{fs[0]:.2f}}}$")

ax.set_xlabel("Number of atoms")
ax.set_ylabel("Inference time (s)")
ax.set_title(f"GPU scaling: ALIGNN-FF vs SlakoNet — {atoms.composition.reduced_formula} supercells")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("alignn_vs_slakonet_scaling.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → alignn_vs_slakonet_scaling.png")

In [ ]:
%matplotlib inline
ax.set_xlabel("Number of atoms")
ax.set_ylabel("Inference time (s)")
ax.set_title(f"GPU scaling: ALIGNN-FF vs SlakoNet — {atoms.composition.reduced_formula} supercells")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()